# How it is asked: full-text criteria in the question instructions instead of in the state

**One factor changed relative to [05_fulltext_criteria.ipynb](05_fulltext_criteria.ipynb)** (review information = only the full-text criteria): the criteria move, word for word, from `state.review` into the `instructions` of the `include` question. The state then holds only the record. No review title, no research questions. Record input and model are unchanged.

| | `state` | `questions.include.instructions` |
|---|---|---|
| criteria only, in state | review: full-text criteria; record: title, abstract | fixed sentence |
| this run | record: title, abstract | fixed sentence + full-text criteria |

Labels: CLEF TAR abstract-level (`label_abs`) and content-level (`label_content`).

Nothing paid runs unless `RUN_SMOKE`, `RUN_SUBSET` or `RUN_PAID` is set to `True` below.

In [1]:
import json, sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
import screening as S

RUN_SMOKE = False    # one paid call
RUN_SUBSET = False   # the 10 reviews with < 1000 records (3,637 records); see the cost estimate
RUN_PAID = False     # all 20 reviews; read the cost estimate first
OUT = S.out_dir("fulltext_criteria_in_question")
COMPARE = {"baseline": S.out_dir("baseline"),
           "criteria_in_state": S.out_dir("review_fulltext_criteria_only"),
           "criteria_in_question": OUT}

/Users/joshua/Desktop/Repositories/Research/decision-model-screener/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

In [2]:
records, CRITERIA = S.load_records()
records = S.add_bm25(records, CRITERIA)
FT = json.loads((ROOT / "data" / "processed" / "tar2019_intervention_fulltext_criteria.json").read_text())
assert set(FT) == set(CRITERIA)
n = records.groupby("review").size()
SUBSET = sorted(n[n < 1000].index)
print("subset:", len(SUBSET), "reviews,", int(n[SUBSET].sum()), "records")

20 reviews, 41996 records, 41183 unique PMIDs
not returned by PubMed: 0 | no abstract: 2183 of 41996


subset: 10 reviews, 3637 records


## Decision model input

In [3]:
def build_body(review, title, abstract):
    base = S.QUESTIONS["include"]["instructions"]
    return {
        "model": S.MODEL,
        "state": {
            "record": {"title": title, "abstract": abstract},
        },
        "questions": {
            "include": {
                "type": "noul",
                "instructions": f"{base}\n\n{FT[review]['criteria']}",
            }
        },
    }

print(json.dumps(build_body(*records.loc[0, ["review", "title", "abstract"]]), indent=1)[:3000])

{
 "model": "typesafe/jev-1.14",
 "state": {
  "record": {
   "title": "Bronchiectasis Severity Is an Independent Risk Factor for Vascular Disease in a Bronchiectasis Cohort.",
   "abstract": "BACKGROUND: Interest in the association of vascular disease with COPD and pneumonia has increased, but there is a lack of research in this area with patients with bronchiectasis.\nMETHODS: A retrospective study of 400 patients attending a specialist bronchiectasis clinic in NHS Lothian (Edinburgh, UK) between May 2013 and September 2014 was conducted. The study assessed the prevalence of vascular disease (ischemic heart disease, cerebrovascular disease, peripheral vascular disease, and atrial fibrillation). Using multivariable models, independent risk factors were identified for vascular disease that developed following the diagnosis of bronchiectasis.\nRESULTS: The study included 400 patients. There was preexisting vascular disease (ie, before the diagnosis of bronchiectasis) in 44 patients (11%

## Cost estimate (no API call)

Same text as the criteria-only run, only moved, so the size should be almost the same. The real cost per call is compared below from the `usage` field.

In [4]:
est = S.estimate_cost(records, build_body)
print(f"subset ({len(SUBSET)} reviews): ${est.loc[SUBSET, 'usd'].sum():.2f}")

TypeSafe | typesafe/jev-1.14-20261014 | USD per 1M input tokens: 0.041999999999999996 | output: 0


requests: 41996 | mean chars per request: 7847 | max: 22869
estimated input tokens: 107.7M | estimated cost: $4.52
subset (10 reviews): $0.27


## Smoke test (one paid call)

Checks that the API accepts long instructions, and compares tokens and cost with the baseline call for the same record.

In [5]:
if RUN_SMOKE:
    rec = records.loc[0, ["review", "pmid", "title", "abstract"]]
    resp = S.call(build_body(rec.review, rec.title, rec.abstract))
    print(json.dumps(resp, indent=1))
    base = {r["pmid"]: r for r in map(json.loads, (COMPARE["criteria_in_state"] / f"{rec.review}.jsonl").open())}[rec.pmid]["response"]
    print("criteria in state, same record:", base["usage"]["input_tokens"], "tokens,", base["usage"]["cost"], "USD, score", base["answers"]["include"]["noul"])

## Subset (paid)

In [6]:
if RUN_SUBSET:
    S.run_reviews(records, SUBSET, build_body, OUT)

## Full run (paid)

In [7]:
if RUN_PAID:
    S.run_reviews(records, CRITERIA, build_body, OUT)

## Results

Both configurations on the same records: the reviews with a cache file for this run. Cost and tokens are summed from the `usage` field of the cached responses for exactly these records.

In [8]:
done = sorted(p.stem for p in OUT.glob("*.jsonl"))
print(len(done), "of", len(CRITERIA), "reviews run")
sub = records[records.review.isin(done)].copy()
for col, path in COMPARE.items():
    sub = S.add_scores(sub, path, col)

results = {c: S.evaluate(sub, c) for c in ("bm25", *COMPARE)}
if done:
    results["in_question - in_state"] = results["criteria_in_question"] - results["criteria_in_state"]
    results["in_question - in_state"]["reviews"] = results["criteria_in_state"]["reviews"]
pd.concat(results)

0 of 20 reviews run
baseline: scored 0 of 0 records | model: {'typesafe/jev-1.14-20261014'} | cost: $1.80 | without a score (ranked last): 0


criteria_in_state: scored 0 of 0 records | model: {'typesafe/jev-1.14-20261014'} | cost: $0.22 | without a score (ranked last): 0
criteria_in_question: scored 0 of 0 records | model: set() | cost: $0.00 | without a score (ranked last): 0


reviews
bm25                 label_abs            0
                     label_content        0
baseline             label_abs            0
                     label_content        0
criteria_in_state    label_abs            0
                     label_content        0
criteria_in_question label_abs            0
                     label_content        0

In [9]:
def usage(path, reviews):
    tok = cost = n = 0
    for r in reviews:
        for row in map(json.loads, (path / f"{r}.jsonl").open()):
            u = row["response"].get("usage", {})
            tok += u.get("input_tokens", 0); cost += u.get("cost") or 0; n += 1
    return {"calls": n, "input_tokens": tok, "usd": round(cost, 3), "tokens_per_call": round(tok / max(n, 1))}

pd.DataFrame({c: usage(p, done) for c, p in COMPARE.items()}).T

,calls,input_tokens,usd,tokens_per_call
baseline,0,0,0,0
criteria_in_state,0,0,0,0
criteria_in_question,0,0,0,0


In [10]:
# per-review MAP
rows = {}
for r, df in sub.groupby("review"):  # empty until a run has finished
    for label in ("label_abs", "label_content"):
        for c in COMPARE:
            m = S.metrics(df.sort_values([c, "bm25", "pmid"], ascending=[False, False, True])[label])
            rows.setdefault(r, {})[(label, c)] = round(m["MAP"] * 100, 1) if m else None
per_review = pd.DataFrame(rows).T
for label in ("label_abs", "label_content") if rows else ():
    d = per_review[(label, "criteria_in_question")] - per_review[(label, "criteria_in_state")]
    print(f"{label}: criteria in question vs in state, better {(d > 0).sum()} / worse {(d < 0).sum()} of {d.notna().sum()} reviews")
per_review

""
